# Contrastive XGBoost — multi-output classification from a `vector` column

Starts from a DataFrame with a **`vector`** column (text embeddings) and a
**`labels`** column (`list` of `"category::label"` keys). One binary XGBoost
is trained on `(text_vec, label_vec)` **pairs** — *does this text match this
label?* — and scored against every label at eval time, so it reads out as a
multi-output classifier.

Preprocessing is plain dataframe manipulation:
1. `embed_labels` — embed each unique label → `labels_df[label_key, vector_label]`.
2. `make_contrastive_dataset` — cross texts × labels → long df of pairs
   (`vector`, `vector_label`, `label` ∈ {0,1}).
3. **In the cell** — build features from those two vector columns with `_combine`,
   then `fit_contrastive_xgb` and `evaluate_multioutput`.

In [ ]:
# ============================== CONFIG ==============================
DATA_PATH = "labeled_demo.csv"          # text,labels -> we embed text into `vector`
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"

params = {"n_estimators": 200, "max_depth": 6, "learning_rate": 0.1}
N_NEG = 3        # negative labels sampled per text
THRESHOLD = 0.5

label_dict = {
    "sentiment": {
        "positive": "expresses approval, happiness, or satisfaction",
        "negative": "expresses disapproval, anger, or frustration",
        "neutral":  "factual or descriptive with no clear emotional valence",
    },
    "topic": {
        "pricing":  "mentions cost, price, fees, value for money",
        "support":  "mentions customer service, help, agents, response time",
        "quality":  "mentions product quality, defects, durability, materials",
        "shipping": "mentions delivery, shipping speed, packaging, arrival",
    },
}
# ===================================================================

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import classification_report, f1_score


def _combine(t, l):
    """Contrastive features for text/label vectors: concat + diff + prod -> (.., 4d)."""
    return np.concatenate([t, l, t - l, t * l], axis=-1)


def embed_labels(label_dict, embed):
    """Step 1: one row per "category::label" key with its embedding.
    Columns: label_key, vector_label."""
    rows = [(f"{c}::{l}", f"{l}. {d}")
            for c, labs in label_dict.items() for l, d in labs.items()]
    vecs = np.asarray(embed.encode([d for _, d in rows], normalize_embeddings=True))
    return pd.DataFrame({"label_key": [k for k, _ in rows], "vector_label": list(vecs)})


def make_contrastive_dataset(df, labels_df, *, labels_col="labels", vector_col="vector",
                             n_neg=N_NEG, seed=42):
    """Step 2: cross every text with every label, then keep all positives plus
    `n_neg` sampled negatives per text. Long dataframe with one (text, label) pair
    per row: vector, vector_label, label (1 = text has this label, 0 = negative).

    ponytail: cross join is n_text*n_labels rows before sampling; fine at this scale.
    """
    keys = set(labels_df["label_key"])
    texts = (df[[vector_col, labels_col]]
             .rename(columns={vector_col: "vector"}).reset_index(drop=True))
    texts = texts[texts[labels_col].apply(lambda labs: any(k in keys for k in labs))].copy()
    texts["_tid"] = range(len(texts))                       # one id per kept text

    pairs = texts.merge(labels_df, how="cross")             # every text × every label
    pairs["label"] = [int(k in labs)                        # 1 if the text carries this label
                      for k, labs in zip(pairs["label_key"], pairs[labels_col])]

    neg = pairs[pairs.label == 0].sample(frac=1, random_state=seed)   # shuffle negatives
    neg = neg[neg.groupby("_tid").cumcount() < n_neg]                 # first n_neg per text
    pairs = pd.concat([pairs[pairs.label == 1], neg]).sort_values("_tid")
    return pairs[["vector", "vector_label", "label", "label_key"]].reset_index(drop=True)


def fit_contrastive_xgb(X, y, **params):
    """One binary XGBoost on the contrastive pairs."""
    model = xgb.XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                              tree_method="hist", random_state=42, **params)
    model.fit(X, y)
    return model


def evaluate_multioutput(model, df, labels_df, *, labels_col="labels",
                         vector_col="vector", threshold=THRESHOLD):
    """Score every text against every label, threshold, report multi-label metrics."""
    keys = labels_df["label_key"].tolist()
    L = np.stack(labels_df["vector_label"])
    T = np.stack([np.asarray(v, np.float32) for v in df[vector_col]])
    n, m = len(T), len(keys)
    # ponytail: naive all-pairs (n*m rows); batch if n*m gets large.
    X = _combine(np.repeat(T, m, axis=0), np.tile(L, (n, 1)))
    proba = model.predict_proba(X)[:, 1].reshape(n, m)
    pred = (proba >= threshold).astype(int)
    ki = {k: i for i, k in enumerate(keys)}
    truth = np.zeros((n, m), int)
    for r, labs in enumerate(df[labels_col]):
        for k in labs:
            if k in ki:
                truth[r, ki[k]] = 1

    pred_b, truth_b = pred.astype(bool), truth.astype(bool)
    tp = (pred_b & truth_b).sum(1)
    fp = (pred_b & ~truth_b).sum(1)
    n_true, n_neg = truth_b.sum(1), (~truth_b).sum(1)
    row_acc = (pred == truth).all(1).mean()                       # whole row exactly right
    rec = np.divide(tp, n_true, out=np.zeros(n), where=n_true > 0)    # % of true labels caught
    fpr = np.divide(fp, n_neg, out=np.zeros(n), where=n_neg > 0)      # % of non-labels wrongly predicted
    avg_correct = rec[n_true > 0].mean() if (n_true > 0).any() else 0.0
    avg_incorrect = fpr[n_neg > 0].mean() if (n_neg > 0).any() else 0.0

    print(classification_report(truth, pred, target_names=keys, zero_division=0))
    print(f"row_accuracy {row_acc:.1%} | avg correct labels {avg_correct:.1%} | "
          f"avg incorrect labels {avg_incorrect:.1%}")
    return {"proba": proba, "pred": pred, "truth": truth,
            "micro_f1": f1_score(truth, pred, average="micro", zero_division=0),
            "macro_f1": f1_score(truth, pred, average="macro", zero_division=0),
            "row_accuracy": row_acc,
            "avg_pct_correct": avg_correct, "avg_pct_incorrect": avg_incorrect}


def _smoke_test():
    """Runs the whole pipeline on random vectors so the logic has a check."""
    rng = np.random.default_rng(0)
    labels_df = pd.DataFrame({
        "label_key": [f"c::{i}" for i in range(5)],
        "vector_label": [rng.normal(size=8).astype(np.float32) for _ in range(5)],
    })
    df = pd.DataFrame({
        "vector": [rng.normal(size=8).astype(np.float32) for _ in range(20)],
        "labels": [list(rng.choice(labels_df.label_key, 2, replace=False)) for _ in range(20)],
    })
    pairs = make_contrastive_dataset(df, labels_df, n_neg=2)
    assert {"vector", "vector_label", "label"} <= set(pairs.columns)
    assert set(pairs["label"].unique()) <= {0, 1}
    X = _combine(np.stack(pairs["vector"]), np.stack(pairs["vector_label"]))  # features outside
    y = pairs["label"].to_numpy()
    assert X.shape == (len(pairs), 8 * 4)
    out = evaluate_multioutput(fit_contrastive_xgb(X, y, n_estimators=10, max_depth=2),
                               df, labels_df)
    assert out["pred"].shape == out["truth"].shape == (20, 5)
    print("smoke ok — micro_f1", round(out["micro_f1"], 3))


_smoke_test()

## Run on real data

Build `label_vectors` from `label_dict` and the starting `vector` column from
`DATA_PATH` (this is the only place text is embedded), then fit and evaluate.

In [ ]:
import json
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split

embed = SentenceTransformer(EMBED_MODEL_NAME)

# 1) embed each unique label
labels_df = embed_labels(label_dict, embed)

# starting frame: a `vector` column (text embedding) + multi-hot `labels` list
raw = pd.read_csv(DATA_PATH)
raw["labels"] = raw["labels"].apply(
    lambda s: json.loads(s) if isinstance(s, str) and s.startswith("[") else [])
raw["vector"] = list(np.asarray(
    embed.encode(raw["text"].tolist(), normalize_embeddings=True)))
df = raw[["vector", "labels"]]

train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)

# 2) contrastive dataset as a dataframe — one (text, label) pair per row
pairs = make_contrastive_dataset(train_df, labels_df)
display(pairs.head())

# 3) features from the pairs dataframe (outside the builder), then fit + evaluate
X = _combine(np.stack(pairs["vector"]), np.stack(pairs["vector_label"]))
y = pairs["label"].to_numpy()
model = fit_contrastive_xgb(X, y, **params)

metrics = evaluate_multioutput(model, test_df, labels_df)
print("micro_f1", round(metrics["micro_f1"], 3), "| macro_f1", round(metrics["macro_f1"], 3))